# Demonstration of BCOfromObservation (BCO)
This notebook shows how to use the `BCOfromObservation` class to perform behavioral cloning from observation-only expert trajectories, with separate random rollouts for training the inverse dynamics model (IDM).

In [1]:
# Install required libraries (if not already installed)
# !pip install imitation stable-baselines3 gymnasium


In [1]:
import numpy as np
import gymnasium as gym
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.data import rollout as rollout_module
# from utils import args
from imitation.algorithms import bco
# from bco import BCOfromObservation, generate_random_demonstrations

# 1. Load expert policy
rng = np.random.default_rng(0)
env = make_vec_env(
    'seals:seals/CartPole-v0', rng=rng,
    post_wrappers=[lambda env, _: RolloutInfoWrapper(env)],
)
expert_policy = load_policy(
    'ppo-huggingface', organization='HumanCompatibleAI',
    env_name='seals:seals/CartPole-v0', venv=expert_env,
)
print('Expert policy ready')

Expert policy ready


In [2]:
# 2. Roll out expert to collect transitions (obs-only)
rollouts_expert = rollout_module.rollout(
    expert_policy, expert_env,
    rollout_module.make_sample_until(min_episodes=50), rng=rng,
)
expert_transitions = rollout_module.flatten_trajectories(rollouts_expert)
print(f"Collected {len(expert_transitions.obs)} expert transitions")

Collected 28000 expert transitions


In [3]:
# 3. Generate random IDM demonstrations
idm_transitions = bco.generate_random_demonstrations(
    env=expert_env, num_episodes=50, rng=rng
)
print(f"Collected {len(idm_transitions.obs)} random IDM transitions")

Collected 28000 random IDM transitions


In [4]:
idm_transitions[0]

{'obs': array([-0.00514559,  0.02989395, -0.02644835, -0.01802154], dtype=float32),
 'acts': np.int64(1),
 'infos': {},
 'next_obs': array([-0.00454771,  0.22538503, -0.02680879, -0.3189305 ], dtype=float32),
 'dones': np.False_}

In [5]:
# 4. Instantiate BCOfromObservation
from gymnasium.spaces import Box, Discrete
env0 = expert_env.envs[0].unwrapped
bco_trainer = bco.BCOfromObservation(
    observation_space=env0.observation_space,
    action_space=env0.action_space,
    rng=rng,
    demonstrations=expert_transitions,
    idm_demonstrations=idm_transitions,
    continuous=False
)
print('BCO agent instantiated')

BCO agent instantiated


In [ ]:
reward_before_training, _ = evaluate_policy(bco_trainer.policy, env0, 10)
print(f"Reward before training: {reward_before_training}")

In [ ]:
# 5. Train BCO
bcoo.max_episodes = 100
bcoo.batch_size = 32
bcoo.M = 200
bcoo.train(log_interval=10)

In [ ]:
# 6. Evaluate learned policy
from stable_baselines3.common.evaluation import evaluate_policy
import torch

class TorchPolicyWrapper:
    def __init__(self, policy): self.policy = policy
    def predict(self, obs, state=None, mask=None):
        obs_t = torch.tensor(obs, dtype=torch.float32)
        a = self.policy(obs_t).detach().numpy()
        return a, None

eval_env = gym.make('CartPole-v0')
reward, std = evaluate_policy(
    TorchPolicyWrapper(bco.policy_net), eval_env, n_eval_episodes=10
)
print(f"Avg reward: {reward} ± {std}")